# E-Commerce Order Analytics System - Mini Project

This notebook is my submission for the intern mini project.
I am generating 4 csv files (orders, order_items, products, customers) with some
messy/dirty data, then I clean it, then I load it into SQLite and write some
SQL queries, and at the end I made a small report tool and some test cases.

## Part 1 - Generating the 4 CSV files

In [1]:
import random
import pandas as pd
from datetime import datetime, timedelta

random.seed(42)   # so data is same every time i run this


### 1.1 customers.csv

In [2]:
num_customers = 500
customer_types = ["REGULAR", "PREMIUM", "VIP"]
first_names = ["Amit","Priya","Rahul","Sneha","Vikram","Neha","Arjun","Pooja","Karan","Divya",
               "Rohit","Anjali","Suresh","Kavya","Manoj","Isha","Sanjay","Riya","Deepak","Meena"]
last_names = ["Sharma","Verma","Gupta","Reddy","Iyer","Nair","Rao","Singh","Patel","Mehta"]

customers = []
for i in range(1, num_customers+1):
    cust_id = "C" + str(1000+i)
    name = random.choice(first_names) + " " + random.choice(last_names)

    # making 2% of the emails bad on purpose
    if random.random() < 0.02:
        bad_type = random.choice(["no_at", "no_domain"])
        if bad_type == "no_at":
            email = name.replace(" ","").lower() + "gmail.com"   # missing @
        else:
            email = name.replace(" ","").lower() + "@"           # missing domain
    else:
        email = name.replace(" ","").lower() + str(i) + "@gmail.com"

    reg_date = datetime(2022,1,1) + timedelta(days=random.randint(0, 900))
    cust_type = random.choices(customer_types, weights=[70,20,10])[0]

    customers.append([cust_id, name, email, reg_date.strftime("%Y-%m-%d"), cust_type])

customers_df = pd.DataFrame(customers, columns=["customer_id","customer_name","email","registration_date","customer_type"])
customers_df.head()


,customer_id,customer_name,email,registration_date,customer_type
0,C1001,Sneha Sharma,snehasharma1@gmail.com,2022-09-08,REGULAR
1,C1002,Sneha Patel,snehapatel2@gmail.com,2023-03-09,REGULAR
2,C1003,Rahul Reddy,rahulreddy3@gmail.com,2023-09-09,REGULAR
3,C1004,Arjun Patel,arjunpatel4@gmail.com,2023-04-05,REGULAR
4,C1005,Amit Gupta,amitgupta5@gmail.com,2022-12-15,REGULAR


### 1.2 products.csv

In [3]:
categories = {
    "Electronics": ["Mobiles","Laptops","Headphones","Cameras"],
    "Clothing": ["Men","Women","Kids","Footwear"],
    "Home": ["Kitchen","Furniture","Decor","Cleaning"],
    "Books": ["Fiction","Non-Fiction","Comics","Academic"]
}

product_names = ["Wireless Mouse","Bluetooth Speaker","Cotton T-Shirt","Running Shoes",
                  "Study Table","Novel Book","Kitchen Mixer","Wall Clock","Notebook Set",
                  "Smart Watch","Winter Jacket","Kids Toy","Water Bottle","Office Chair"]

num_products = 500
products = []
for i in range(1, num_products+1):
    prod_id = "P" + str(2000+i)
    cat = random.choice(list(categories.keys()))
    subcat = random.choice(categories[cat])
    name = random.choice(product_names)

    # some names messy on purpose - extra spaces / weird case
    if random.random() < 0.1:
        name = "  " + name.upper() + "  "
    elif random.random() < 0.2:
        name = name.lower()

    cost = round(random.uniform(50, 5000), 2)
    products.append([prod_id, name, cat, subcat, cost])

products_df = pd.DataFrame(products, columns=["product_id","product_name","category","subcategory","cost_price"])
products_df.head()


,product_id,product_name,category,subcategory,cost_price
0,P2001,Wireless Mouse,Books,Fiction,4888.52
1,P2002,water bottle,Electronics,Cameras,4594.34
2,P2003,Running Shoes,Clothing,Women,1804.48
3,P2004,Novel Book,Books,Fiction,1253.88
4,P2005,winter jacket,Clothing,Kids,4958.79


### 1.3 orders.csv

In [4]:
statuses = ["PLACED","SHIPPED","DELIVERED","CANCELLED","RETURNED"]
regions = ["NORTH","SOUTH","EAST","WEST"]

num_orders = 800
customer_ids = customers_df["customer_id"].tolist()

orders = []
for i in range(1, num_orders+1):
    order_id = "O" + str(5000+i)

    # 5% missing customer id
    if random.random() < 0.05:
        cust_id = ""
    else:
        cust_id = random.choice(customer_ids)

    order_dt = datetime(2024,1,1) + timedelta(days=random.randint(0, 550), hours=random.randint(0,23))

    # some dates in wrong format DD-MM-YYYY instead of YYYY-MM-DD HH:MM:SS
    if random.random() < 0.05:
        date_str = order_dt.strftime("%d-%m-%Y")
    else:
        date_str = order_dt.strftime("%Y-%m-%d %H:%M:%S")

    status = random.choices(statuses, weights=[15,20,45,10,10])[0]
    region = random.choice(regions)

    orders.append([order_id, cust_id, date_str, status, region])

orders_df = pd.DataFrame(orders, columns=["order_id","customer_id","order_date","status","region_code"])
orders_df.head()


,order_id,customer_id,order_date,status,region_code
0,O5001,C1276,09-02-2025,DELIVERED,NORTH
1,O5002,C1226,2025-02-11 09:00:00,SHIPPED,SOUTH
2,O5003,C1314,2024-02-25 22:00:00,PLACED,WEST
3,O5004,C1257,2024-04-25 05:00:00,RETURNED,SOUTH
4,O5005,C1166,2025-04-06 00:00:00,DELIVERED,SOUTH


### 1.4 order_items.csv

In [5]:
order_ids = orders_df["order_id"].tolist()
product_ids = products_df["product_id"].tolist()

order_items = []
item_counter = 1
for order_id in order_ids:
    # each order has 1 to 4 items
    n_items = random.randint(1,4)
    for _ in range(n_items):
        item_id = "I" + str(item_counter)
        item_counter += 1
        prod_id = random.choice(product_ids)

        qty = random.randint(1,5)
        # 3% negative qty = returns
        if random.random() < 0.03:
            qty = -qty

        unit_price = round(random.uniform(100, 6000), 2)
        discount = random.choice([0,0,0,5,10,15,20,25,50])

        order_items.append([item_id, order_id, prod_id, qty, unit_price, discount])

order_items_df = pd.DataFrame(order_items, columns=["item_id","order_id","product_id","quantity","unit_price","discount_percent"])
print("total order items rows:", len(order_items_df))
order_items_df.head()


total order items rows: 1977


,item_id,order_id,product_id,quantity,unit_price,discount_percent
0,I1,O5001,P2222,4,4775.90,15
1,I2,O5002,P2057,1,4144.61,25
2,I3,O5002,P2349,2,1809.71,50
3,I4,O5002,P2341,2,1800.73,10
4,I5,O5003,P2233,1,4970.28,0


In [6]:
# saving the raw (dirty) files
customers_df.to_csv("customers.csv", index=False)
products_df.to_csv("products.csv", index=False)
orders_df.to_csv("orders.csv", index=False)
order_items_df.to_csv("order_items.csv", index=False)
print("saved all 4 raw csv files")


saved all 4 raw csv files


## Part 2 - Cleaning the Data

In [7]:
orders_raw = pd.read_csv("orders.csv")
order_items_raw = pd.read_csv("order_items.csv")
products_raw = pd.read_csv("products.csv")
customers_raw = pd.read_csv("customers.csv")


### clean_orders()

In [8]:
def clean_orders(df):
    df = df.copy()
    fixed_dates = 0

    def fix_date(x):
        nonlocal fixed_dates
        x = str(x)
        # try normal format first
        try:
            datetime.strptime(x, "%Y-%m-%d %H:%M:%S")
            return x
        except:
            pass
        # try DD-MM-YYYY format
        try:
            d = datetime.strptime(x, "%d-%m-%Y")
            fixed_dates += 1
            return d.strftime("%Y-%m-%d %H:%M:%S")
        except:
            return None

    df["order_date"] = df["order_date"].apply(fix_date)

    # fill missing customer id
    missing_before = df["customer_id"].isna().sum() + (df["customer_id"]=="").sum()
    df["customer_id"] = df["customer_id"].replace("", "UNKNOWN")
    df["customer_id"] = df["customer_id"].fillna("UNKNOWN")

    print("dates fixed (wrong format):", fixed_dates)
    print("missing customer_id filled with UNKNOWN:", missing_before)
    return df

orders_clean = clean_orders(orders_raw)
orders_clean.head()


dates fixed (wrong format): 33
missing customer_id filled with UNKNOWN: 39


,order_id,customer_id,order_date,status,region_code
0,O5001,C1276,2025-02-09 00:00:00,DELIVERED,NORTH
1,O5002,C1226,2025-02-11 09:00:00,SHIPPED,SOUTH
2,O5003,C1314,2024-02-25 22:00:00,PLACED,WEST
3,O5004,C1257,2024-04-25 05:00:00,RETURNED,SOUTH
4,O5005,C1166,2025-04-06 00:00:00,DELIVERED,SOUTH


### clean_products()

In [9]:
def clean_products(df):
    df = df.copy()
    df["product_name"] = df["product_name"].str.strip().str.title()
    return df

products_clean = clean_products(products_raw)
products_clean.head()


,product_id,product_name,category,subcategory,cost_price
0,P2001,Wireless Mouse,Books,Fiction,4888.52
1,P2002,Water Bottle,Electronics,Cameras,4594.34
2,P2003,Running Shoes,Clothing,Women,1804.48
3,P2004,Novel Book,Books,Fiction,1253.88
4,P2005,Winter Jacket,Clothing,Kids,4958.79


### validate_emails()

In [10]:
def validate_emails(df):
    bad_ids = []
    for _, row in df.iterrows():
        email = str(row["email"])
        if "@" not in email or "." not in email.split("@")[-1]:
            bad_ids.append(row["customer_id"])
    return bad_ids

bad_emails = validate_emails(customers_raw)
print("number of customers with bad email:", len(bad_emails))
bad_emails[:10]


number of customers with bad email: 11


['C1088',
 'C1122',
 'C1141',
 'C1226',
 'C1242',
 'C1266',
 'C1328',
 'C1332',
 'C1413',
 'C1434']

### check_referential_integrity()

In [11]:
def check_referential_integrity(orders_df, order_items_df):
    valid_orders = set(orders_df["order_id"])
    bad_rows = order_items_df[~order_items_df["order_id"].isin(valid_orders)]
    return bad_rows

bad_items = check_referential_integrity(orders_clean, order_items_raw)
print("order_items with order_id that doesn't exist in orders:", len(bad_items))


order_items with order_id that doesn't exist in orders: 0


In [12]:
# saving cleaned files + a small text report
orders_clean.to_csv("orders_clean.csv", index=False)
products_clean.to_csv("products_clean.csv", index=False)
order_items_clean = order_items_raw.copy()
order_items_clean.to_csv("order_items_clean.csv", index=False)
customers_raw.to_csv("customers_clean.csv", index=False)

with open("cleaning_report.txt","w") as f:
    f.write("CLEANING REPORT\n")
    f.write("================\n")
    f.write("Customers with bad email: " + str(len(bad_emails)) + "\n")
    f.write("Order items with bad order_id: " + str(len(bad_items)) + "\n")
    f.write("Negative quantity rows (returns): " + str((order_items_raw['quantity']<0).sum()) + "\n")

print("cleaning done, report saved")


cleaning done, report saved


## Part 3 - SQL Analysis (SQLite)

In [13]:
import sqlite3

conn = sqlite3.connect("shop.db")

orders_clean.to_sql("orders", conn, if_exists="replace", index=False)
order_items_clean.to_sql("order_items", conn, if_exists="replace", index=False)
products_clean.to_sql("products", conn, if_exists="replace", index=False)
customers_raw.to_sql("customers", conn, if_exists="replace", index=False)

print("tables loaded into sqlite")


tables loaded into sqlite


**Q1. Total revenue per category**

In [14]:
q1 = """
SELECT p.category,
       SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)) AS revenue
FROM order_items oi
JOIN products p ON p.product_id = oi.product_id
WHERE oi.quantity > 0
GROUP BY p.category
ORDER BY revenue DESC;
"""
pd.read_sql(q1, conn)


,category,revenue
0,Home,4.295441e+06
1,Electronics,3.761874e+06
2,Clothing,3.706617e+06
3,Books,3.310560e+06


**Q2. Top 10 customers by total order value**

In [15]:
q2 = """
SELECT o.customer_id,
       SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)) AS total_value
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
WHERE oi.quantity > 0
GROUP BY o.customer_id
ORDER BY total_value DESC
LIMIT 10;
"""
pd.read_sql(q2, conn)


,customer_id,total_value
0,UNKNOWN,733863.2235
1,C1275,182482.7850
2,C1238,154780.5045
3,C1300,152114.8355
4,C1438,148488.7120
5,C1037,139223.0205
6,C1152,129240.6455
7,C1291,121156.3550
8,C1088,120340.4310
9,C1052,115153.6440


**Q3. Month-wise order count**

In [16]:
q3 = """
SELECT substr(order_date,1,7) AS month, COUNT(DISTINCT order_id) AS order_count
FROM orders
GROUP BY month
ORDER BY month;
"""
pd.read_sql(q3, conn)


,month,order_count
0,2024-01,36
1,2024-02,40
2,2024-03,39
3,2024-04,47
4,2024-05,63
5,2024-06,42
6,2024-07,44
7,2024-08,38
8,2024-09,32
9,2024-10,37


**Q4. Customers who placed orders but never had an item delivered**

In [17]:
q4 = """
SELECT DISTINCT o.customer_id
FROM orders o
WHERE o.customer_id NOT IN (
    SELECT customer_id FROM orders WHERE status = 'DELIVERED'
);
"""
pd.read_sql(q4, conn).head(10)


,customer_id
0,C1314
1,C1257
2,C1047
3,C1075
4,C1093
5,C1430
6,C1420
7,C1409
8,C1455
9,C1255


**Q5. Products with more returns than purchases**

In [18]:
q5 = """
SELECT product_id,
       SUM(CASE WHEN quantity > 0 THEN quantity ELSE 0 END) AS purchased,
       SUM(CASE WHEN quantity < 0 THEN -quantity ELSE 0 END) AS returned
FROM order_items
GROUP BY product_id
HAVING returned > purchased;
"""
pd.read_sql(q5, conn)


,product_id,purchased,returned
0,P2057,4,5
1,P2300,0,5
2,P2320,3,5
3,P2383,3,4


**Q6. Return rate per category**

In [19]:
q6 = """
SELECT p.category,
       SUM(CASE WHEN oi.quantity < 0 THEN -oi.quantity ELSE 0 END) * 1.0 /
       SUM(ABS(oi.quantity)) AS return_rate
FROM order_items oi
JOIN products p ON p.product_id = oi.product_id
GROUP BY p.category;
"""
pd.read_sql(q6, conn)


,category,return_rate
0,Books,0.029208
1,Clothing,0.032861
2,Electronics,0.032997
3,Home,0.036254


**Q7. Running total of revenue per region (window function)**

In [20]:
q7 = """
SELECT region_code, order_date, daily_revenue,
       SUM(daily_revenue) OVER (PARTITION BY region_code ORDER BY order_date) AS running_total
FROM (
    SELECT o.region_code, substr(o.order_date,1,10) AS order_date,
           SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)) AS daily_revenue
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    WHERE oi.quantity > 0
    GROUP BY o.region_code, substr(o.order_date,1,10)
)
ORDER BY region_code, order_date;
"""
pd.read_sql(q7, conn).head(15)


,region_code,order_date,daily_revenue,running_total
0,EAST,2024-01-01,9030.4875,9030.4875
1,EAST,2024-01-02,13651.0340,22681.5215
2,EAST,2024-01-08,67420.0435,90101.5650
3,EAST,2024-01-15,46826.5150,136928.0800
4,EAST,2024-01-21,20148.0860,157076.1660
5,EAST,2024-01-22,33003.8790,190080.0450
6,EAST,2024-01-24,43804.3650,233884.4100
7,EAST,2024-01-28,24848.0575,258732.4675
8,EAST,2024-02-01,19973.0900,278705.5575
9,EAST,2024-02-02,1709.3000,280414.8575


**Q8. Rank products within category by revenue (DENSE_RANK)**

In [21]:
q8 = """
SELECT category, product_name, total_revenue,
       DENSE_RANK() OVER (PARTITION BY category ORDER BY total_revenue DESC) AS rank_in_category
FROM (
    SELECT p.category, p.product_name,
           SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)) AS total_revenue
    FROM order_items oi
    JOIN products p ON p.product_id = oi.product_id
    WHERE oi.quantity > 0
    GROUP BY p.category, p.product_name
)
ORDER BY category, rank_in_category;
"""
pd.read_sql(q8, conn).head(15)


,category,product_name,total_revenue,rank_in_category
0,Books,Bluetooth Speaker,350474.8205,1
1,Books,Winter Jacket,344784.5770,2
2,Books,Office Chair,317984.7705,3
3,Books,Kitchen Mixer,308021.2620,4
4,Books,Cotton T-Shirt,293710.9465,5
5,Books,Smart Watch,282623.4950,6
6,Books,Water Bottle,248792.2010,7
7,Books,Novel Book,219989.2970,8
8,Books,Kids Toy,211006.8725,9
9,Books,Study Table,191752.3710,10


**Q9. Days between consecutive orders per customer (LAG)**

In [22]:
q9 = """
SELECT customer_id, order_date, previous_order_date,
       julianday(order_date) - julianday(previous_order_date) AS days_gap,
       CASE WHEN julianday(order_date) - julianday(previous_order_date) > 30
            THEN 'At Risk' ELSE 'OK' END AS risk_flag
FROM (
    SELECT customer_id, order_date,
           LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) AS previous_order_date
    FROM orders
    WHERE customer_id != 'UNKNOWN'
)
WHERE previous_order_date IS NOT NULL
ORDER BY customer_id, order_date;
"""
pd.read_sql(q9, conn).head(15)


,customer_id,order_date,previous_order_date,days_gap,risk_flag
0,C1002,2025-01-28 22:00:00,2024-06-12 18:00:00,230.166667,At Risk
1,C1004,2024-05-26 14:00:00,2024-03-07 15:00:00,79.958333,At Risk
2,C1004,2024-06-21 00:00:00,2024-05-26 14:00:00,25.416667,OK
3,C1004,2024-09-08 15:00:00,2024-06-21 00:00:00,79.625000,At Risk
4,C1004,2025-04-16 10:00:00,2024-09-08 15:00:00,219.791667,At Risk
5,C1005,2024-04-18 17:00:00,2024-04-15 20:00:00,2.875000,OK
6,C1005,2025-06-14 18:00:00,2024-04-18 17:00:00,422.041667,At Risk
7,C1008,2025-06-10 03:00:00,2024-12-11 06:00:00,180.875000,At Risk
8,C1009,2025-05-13 00:00:00,2024-02-07 10:00:00,460.583333,At Risk
9,C1009,2025-07-04 21:00:00,2025-05-13 00:00:00,52.875000,At Risk


**Q10. Monthly revenue per customer -> category (High/Medium/Low) -> count per month (CTE)**

In [23]:
q10 = """
WITH monthly_rev AS (
    SELECT o.customer_id, substr(o.order_date,1,7) AS month,
           SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)) AS revenue
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    WHERE oi.quantity > 0
    GROUP BY o.customer_id, month
),
categorized AS (
    SELECT month, customer_id,
           CASE WHEN revenue > 10000 THEN 'High'
                WHEN revenue >= 5000 THEN 'Medium'
                ELSE 'Low' END AS revenue_category
    FROM monthly_rev
)
SELECT month, revenue_category, COUNT(*) AS customer_count
FROM categorized
GROUP BY month, revenue_category
ORDER BY month;
"""
pd.read_sql(q10, conn).head(15)


,month,revenue_category,customer_count
0,2024-01,High,24
1,2024-01,Low,3
2,2024-01,Medium,5
3,2024-02,High,27
4,2024-02,Low,6
5,2024-02,Medium,6
6,2024-03,High,29
7,2024-03,Low,4
8,2024-03,Medium,4
9,2024-04,High,30


**Q11. Customer quartiles by lifetime value (NTILE)**

In [24]:
q11 = """
WITH ltv AS (
    SELECT o.customer_id,
           SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)) AS total_value
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    WHERE oi.quantity > 0 AND o.customer_id != 'UNKNOWN'
    GROUP BY o.customer_id
)
SELECT customer_id, total_value,
       NTILE(4) OVER (ORDER BY total_value DESC) AS quartile,
       CASE NTILE(4) OVER (ORDER BY total_value DESC)
            WHEN 1 THEN 'Platinum'
            WHEN 2 THEN 'Gold'
            WHEN 3 THEN 'Silver'
            ELSE 'Bronze' END AS quartile_label
FROM ltv
ORDER BY total_value DESC;
"""
pd.read_sql(q11, conn).head(15)


,customer_id,total_value,quartile,quartile_label
0,C1275,182482.7850,1,Platinum
1,C1238,154780.5045,1,Platinum
2,C1300,152114.8355,1,Platinum
3,C1438,148488.7120,1,Platinum
4,C1037,139223.0205,1,Platinum
5,C1152,129240.6455,1,Platinum
6,C1291,121156.3550,1,Platinum
7,C1088,120340.4310,1,Platinum
8,C1052,115153.6440,1,Platinum
9,C1091,111813.6770,1,Platinum


**Q12. Year over year revenue comparison**

In [25]:
q12 = """
WITH monthly AS (
    SELECT substr(order_date,1,4) AS year, substr(order_date,6,2) AS month,
           SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)) AS revenue
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    WHERE oi.quantity > 0
    GROUP BY year, month
)
SELECT a.year, a.month, a.revenue,
       b.revenue AS prev_year_revenue,
       CASE WHEN b.revenue IS NULL OR b.revenue = 0 THEN NULL
            ELSE ROUND((a.revenue - b.revenue) * 100.0 / b.revenue, 2) END AS yoy_growth_percent
FROM monthly a
LEFT JOIN monthly b ON b.month = a.month AND CAST(b.year AS INTEGER) = CAST(a.year AS INTEGER) - 1
ORDER BY a.year, a.month;
"""
pd.read_sql(q12, conn)


,year,month,revenue,prev_year_revenue,yoy_growth_percent
0,2024,01,8.187646e+05,NaN,NaN
1,2024,02,7.151633e+05,NaN,NaN
2,2024,03,8.088932e+05,NaN,NaN
3,2024,04,8.213365e+05,NaN,NaN
4,2024,05,1.277048e+06,NaN,NaN
5,2024,06,9.115828e+05,NaN,NaN
6,2024,07,6.545169e+05,NaN,NaN
7,2024,08,8.215486e+05,NaN,NaN
8,2024,09,6.318754e+05,NaN,NaN
9,2024,10,7.557505e+05,NaN,NaN


**Q13. First vs most recent purchased category per customer**

In [26]:
q13 = """
WITH cat_orders AS (
    SELECT o.customer_id, o.order_date, p.category,
           ROW_NUMBER() OVER (PARTITION BY o.customer_id ORDER BY o.order_date ASC) AS rn_first,
           ROW_NUMBER() OVER (PARTITION BY o.customer_id ORDER BY o.order_date DESC) AS rn_last
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    JOIN products p ON p.product_id = oi.product_id
    WHERE o.customer_id != 'UNKNOWN'
),
first_cat AS (SELECT customer_id, category AS first_category FROM cat_orders WHERE rn_first = 1),
last_cat AS (SELECT customer_id, category AS last_category FROM cat_orders WHERE rn_last = 1)
SELECT f.customer_id, f.first_category, l.last_category,
       CASE WHEN f.first_category != l.last_category THEN 'Yes' ELSE 'No' END AS category_shift
FROM first_cat f
JOIN last_cat l ON l.customer_id = f.customer_id;
"""
pd.read_sql(q13, conn).head(15)


,customer_id,first_category,last_category,category_shift
0,C1002,Clothing,Electronics,Yes
1,C1003,Electronics,Electronics,No
2,C1004,Clothing,Electronics,Yes
3,C1005,Books,Home,Yes
4,C1006,Electronics,Electronics,No
5,C1008,Clothing,Home,Yes
6,C1009,Clothing,Electronics,Yes
7,C1011,Home,Electronics,Yes
8,C1012,Books,Home,Yes
9,C1013,Electronics,Books,Yes


**Q14. Cumulative revenue % (top N% of customers)**

In [27]:
q14 = """
WITH cust_rev AS (
    SELECT o.customer_id,
           SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)) AS revenue
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    WHERE oi.quantity > 0 AND o.customer_id != 'UNKNOWN'
    GROUP BY o.customer_id
)
SELECT customer_id, revenue,
       SUM(revenue) OVER (ORDER BY revenue DESC) AS cumulative_revenue,
       ROUND(SUM(revenue) OVER (ORDER BY revenue DESC) * 100.0 / SUM(revenue) OVER (), 2) AS cumulative_percent
FROM cust_rev
ORDER BY revenue DESC;
"""
pd.read_sql(q14, conn).head(15)


,customer_id,revenue,cumulative_revenue,cumulative_percent
0,C1275,182482.7850,1.824828e+05,1.27
1,C1238,154780.5045,3.372633e+05,2.35
2,C1300,152114.8355,4.893781e+05,3.41
3,C1438,148488.7120,6.378668e+05,4.45
4,C1037,139223.0205,7.770899e+05,5.42
5,C1152,129240.6455,9.063305e+05,6.32
6,C1291,121156.3550,1.027487e+06,7.16
7,C1088,120340.4310,1.147827e+06,8.00
8,C1052,115153.6440,1.262981e+06,8.81
9,C1091,111813.6770,1.374795e+06,9.59


**Q15. Cohort analysis (registration month vs order month)**

In [28]:
q15 = """
WITH cohort AS (
    SELECT customer_id, substr(registration_date,1,7) AS cohort_month
    FROM customers
),
orders_month AS (
    SELECT customer_id, substr(order_date,1,7) AS order_month
    FROM orders
    WHERE customer_id != 'UNKNOWN'
)
SELECT c.cohort_month,
       SUM(CASE WHEN om.order_month = c.cohort_month THEN 1 ELSE 0 END) AS month_0,
       SUM(CASE WHEN om.order_month = strftime('%Y-%m', c.cohort_month || '-01', '+1 month') THEN 1 ELSE 0 END) AS month_1,
       SUM(CASE WHEN om.order_month = strftime('%Y-%m', c.cohort_month || '-01', '+2 month') THEN 1 ELSE 0 END) AS month_2,
       SUM(CASE WHEN om.order_month = strftime('%Y-%m', c.cohort_month || '-01', '+3 month') THEN 1 ELSE 0 END) AS month_3
FROM cohort c
LEFT JOIN orders_month om ON om.customer_id = c.customer_id
GROUP BY c.cohort_month
ORDER BY c.cohort_month;
"""
pd.read_sql(q15, conn).head(15)


,cohort_month,month_0,month_1,month_2,month_3
0,2022-01,0,0,0,0
1,2022-02,0,0,0,0
2,2022-03,0,0,0,0
3,2022-04,0,0,0,0
4,2022-05,0,0,0,0
5,2022-06,0,0,0,0
6,2022-07,0,0,0,0
7,2022-08,0,0,0,0
8,2022-09,0,0,0,0
9,2022-10,0,0,0,0


**Q16. Products frequently bought together (self join)**

In [29]:
q16 = """
SELECT a.product_id AS product_a, b.product_id AS product_b, COUNT(*) AS times_bought_together
FROM order_items a
JOIN order_items b ON a.order_id = b.order_id AND a.product_id < b.product_id
GROUP BY a.product_id, b.product_id
ORDER BY times_bought_together DESC
LIMIT 15;
"""
pd.read_sql(q16, conn)


,product_a,product_b,times_bought_together
0,P2020,P2173,2
1,P2049,P2100,2
2,P2050,P2250,2
3,P2060,P2230,2
4,P2074,P2441,2
5,P2075,P2250,2
6,P2169,P2311,2
7,P2175,P2343,2
8,P2190,P2272,2
9,P2207,P2441,2


## Part 4 - Simple Command Line Report Tool

In [30]:
def generate_report(start_date, end_date, prev_start, prev_end):
    q_current = f"""
    SELECT COUNT(DISTINCT o.order_id) AS total_orders,
           SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)) AS revenue,
           COUNT(DISTINCT o.customer_id) AS unique_customers
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    WHERE oi.quantity > 0 AND substr(o.order_date,1,10) BETWEEN '{start_date}' AND '{end_date}';
    """
    current = pd.read_sql(q_current, conn).iloc[0]

    q_prev = f"""
    SELECT SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)) AS revenue
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    WHERE oi.quantity > 0 AND substr(o.order_date,1,10) BETWEEN '{prev_start}' AND '{prev_end}';
    """
    prev_revenue = pd.read_sql(q_prev, conn).iloc[0]["revenue"]

    q_top3 = f"""
    SELECT p.product_name, SUM(oi.quantity) AS units_sold
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    JOIN products p ON p.product_id = oi.product_id
    WHERE oi.quantity > 0 AND substr(o.order_date,1,10) BETWEEN '{start_date}' AND '{end_date}'
    GROUP BY p.product_name
    ORDER BY units_sold DESC
    LIMIT 3;
    """
    top3 = pd.read_sql(q_top3, conn)

    print("REPORT for", start_date, "to", end_date)
    print("Total orders:", current["total_orders"])
    print("Revenue:", round(current["revenue"],2) if current["revenue"] else 0)
    print("Unique customers:", current["unique_customers"])

    if prev_revenue:
        change = round((current["revenue"] - prev_revenue) * 100 / prev_revenue, 2)
        print("Change vs previous period:", change, "%")
    else:
        print("No data for previous period to compare")

    print("Top 3 products:")
    print(top3)

# just calling it once with some sample dates instead of input(), so the notebook runs by itself
generate_report("2024-06-01","2024-06-30","2024-05-01","2024-05-31")


REPORT for 2024-06-01 to 2024-06-30
Total orders: 42.0
Revenue: 911582.81
Unique customers: 41.0
Change vs previous period: -28.62 %
Top 3 products:
    product_name  units_sold
0  Kitchen Mixer          38
1   Office Chair          37
2    Study Table          30


*(If running this as a plain `.py` script from the terminal, `report_type` and dates can be taken using `input()` and then passed into `generate_report()`.)*

## Part 5 - Edge Case Tests

In [31]:
def test_order_id_not_in_orders():
    bad = check_referential_integrity(orders_clean, order_items_raw)
    print("Test 1 - order_items with unknown order_id:", len(bad), "rows found")

def test_discount_above_100():
    bad = order_items_raw[order_items_raw["discount_percent"] > 100]
    print("Test 2 - discount_percent > 100:", len(bad), "rows found")

def test_zero_quantity():
    zero_qty = order_items_raw[order_items_raw["quantity"] == 0]
    print("Test 3 - rows with quantity = 0:", len(zero_qty))

def test_future_order_date():
    today = datetime.now()
    future = orders_clean[pd.to_datetime(orders_clean["order_date"], errors="coerce") > today]
    print("Test 4 - orders with a future date:", len(future))

test_order_id_not_in_orders()
test_discount_above_100()
test_zero_quantity()
test_future_order_date()


Test 1 - order_items with unknown order_id: 0 rows found
Test 2 - discount_percent > 100: 0 rows found
Test 3 - rows with quantity = 0: 0
Test 4 - orders with a future date: 0


In [32]:
conn.close()
print("done")


done
